# 02 — Data Preprocessing Pipeline

All encoding, feature engineering, train/test split, and scaling in one place. Saves CSVs for downstream notebooks.

## Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import os, pickle, json

os.makedirs('../outputs', exist_ok=True)

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
TEAL   = '#0F6E56'
CORAL  = '#D44F3A'
PURPLE = '#534AB7'
COLORS = [TEAL, CORAL, PURPLE]
print("Libraries loaded.")


Libraries loaded.


## Load Raw Data

In [2]:
df = pd.read_csv('../data/telco_churn.csv')
print(f"Raw dataset: {df.shape}")
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
print("TotalCharges fixed.")


Raw dataset: (7043, 21)
TotalCharges fixed.


## Encoding

In [3]:
# Target
df['Churn_binary'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Binary columns
df['gender'] = df['gender'].map({'Male': 1, 'Female': 0})
for col in ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

# Service columns
for col in ['MultipleLines','OnlineSecurity','OnlineBackup',
            'DeviceProtection','TechSupport','StreamingTV','StreamingMovies']:
    df[col] = df[col].map({'Yes': 1, 'No': 0,
                           'No phone service': 0, 'No internet service': 0})
print("All binary columns encoded.")


All binary columns encoded.


## Feature Engineering

Three engineered features — each with a business justification:

In [4]:
# Feature 1: Charges Per Month — normalises TotalCharges for tenure
# WHY: Raw TotalCharges is confounded by tenure. charges_per_month captures
# SPENDING INTENSITY — how much a customer pays relative to their tenure.
df['charges_per_month'] = df['TotalCharges'] / (df['tenure'] + 1)

# Feature 2: High Value New — the highest-risk segment flag
# WHY: New customers (<=12m) paying premium prices ($70+/month) have not
# built loyalty but are questioning value-for-money. Churn rate ~50%.
df['high_value_new'] = ((df['tenure'] <= 12) & (df['MonthlyCharges'] > 70)).astype(int)

# Feature 3: Num Services — switching cost proxy
# WHY: A customer with 5 add-on services faces significant hassle to
# replicate that bundle elsewhere. Each additional service reduces churn.
service_cols = ['OnlineSecurity','OnlineBackup','DeviceProtection',
                'TechSupport','StreamingTV','StreamingMovies']
df['num_services'] = df[service_cols].sum(axis=1)

print("Feature engineering complete.")
print(f"  charges_per_month — mean: {df['charges_per_month'].mean():.2f}")
print(f"  high_value_new    — {df['high_value_new'].mean()*100:.1f}% flagged")
print(f"  num_services      — mean: {df['num_services'].mean():.2f}")


Feature engineering complete.
  charges_per_month — mean: 58.99
  high_value_new    — 12.4% flagged
  num_services      — mean: 2.04


## One-Hot Encoding & Final Feature Matrix

In [5]:
df_model = pd.get_dummies(df.copy(),
               columns=['Contract', 'PaymentMethod', 'InternetService'],
               drop_first=True)
df_model = df_model.drop(['customerID', 'Churn'], axis=1, errors='ignore')

print(f"Final feature matrix: {df_model.shape}")
print(f"Churn rate: {df_model['Churn_binary'].mean():.3f}")
print(f"\nColumns: {df_model.columns.tolist()}")


Final feature matrix: (7043, 27)
Churn rate: 0.265

Columns: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'Churn_binary', 'charges_per_month', 'high_value_new', 'num_services', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check', 'InternetService_Fiber optic', 'InternetService_No']


## Train / Test Split (Stratified 80/20)

In [6]:
from sklearn.model_selection import train_test_split

X = df_model.drop('Churn_binary', axis=1)
y = df_model['Churn_binary']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Train churn rate: {y_train.mean():.3f} | Test churn rate: {y_test.mean():.3f}")
print("Stratified split preserves 27/73 class ratio in both sets. ✓")


Train: (5634, 26) | Test: (1409, 26)
Train churn rate: 0.265 | Test churn rate: 0.265
Stratified split preserves 27/73 class ratio in both sets. ✓


## Standard Scaling (for Logistic Regression)

Tree models (RF, XGBoost) don't need scaling. We keep both scaled and unscaled versions.

In [7]:
from sklearn.preprocessing import StandardScaler

numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges',
                  'charges_per_month', 'num_services']
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()
X_train_scaled[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test_scaled[numerical_cols]  = scaler.transform(X_test[numerical_cols])

print("Scaling complete. Numerical columns standardised for Logistic Regression.")
print(f"Scaled columns: {numerical_cols}")


Scaling complete. Numerical columns standardised for Logistic Regression.
Scaled columns: ['tenure', 'MonthlyCharges', 'TotalCharges', 'charges_per_month', 'num_services']


## Save All Files

In [8]:
X_train.to_csv('../data/X_train.csv', index=False)
X_test.to_csv('../data/X_test.csv', index=False)
X_train_scaled.to_csv('../data/X_train_scaled.csv', index=False)
X_test_scaled.to_csv('../data/X_test_scaled.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)
print("All preprocessed files saved to ../data/")
print(f"  X_train.csv        — {X_train.shape}")
print(f"  X_test.csv         — {X_test.shape}")
print(f"  X_train_scaled.csv — {X_train_scaled.shape}")
print(f"  X_test_scaled.csv  — {X_test_scaled.shape}")
print(f"  y_train.csv        — {y_train.shape}")
print(f"  y_test.csv         — {y_test.shape}")


All preprocessed files saved to ../data/
  X_train.csv        — (5634, 26)
  X_test.csv         — (1409, 26)
  X_train_scaled.csv — (5634, 26)
  X_test_scaled.csv  — (1409, 26)
  y_train.csv        — (5634,)
  y_test.csv         — (1409,)


## Pipeline Summary

This notebook produces all data artefacts consumed by downstream notebooks. The pipeline design decisions are:

| Decision | Choice | Justification |
|---|---|---|
| Train/test split ratio | 80/20 | Standard for ~7,000 row datasets — enough test samples (1,409) for stable metric estimates |
| Stratified split | Yes (`stratify=y`) | Preserves the 27/73 class imbalance in both sets — prevents a lucky split from inflating evaluation metrics |
| Scaling strategy | Numerical columns only, fit on train only | Categorical binary columns don't need scaling. Fitting only on train data prevents data leakage from test set statistics entering the scaler |
| Two versions kept (scaled + unscaled) | `X_train` and `X_train_scaled` | Logistic Regression requires scaled input for coefficient comparability; tree models (RF, XGBoost) are scale-invariant and perform better without scaling |
| `drop_first=True` in one-hot encoding | Yes | Drops one category per feature to avoid perfect multicollinearity (the dummy variable trap), which would cause singular matrices in Logistic Regression |

**Data flow**: All 6 CSVs saved here → loaded by Notebooks 03, 04, 05, 06, 07, 08.